# Module 2 — Exploratory Data Analysis
Data Analyst role. EDA is run on the **integrated, not yet cleaned** data (real OULAD + raw synthetic logs), so
that anomalies discovered here motivate the cleaning rules in Module 3.

Contents
1. Univariate distributions, skewness and outliers (IQR and z-score rules, Lecture 7)
2. Correlations among engagement and performance variables
3. Comparisons across **course type**, **cohort** (presentation) and **enrollment period**
4. Required relationships: login frequency vs final score; video completion vs submission timeliness
5. Behaviour vs final outcome
6. Time-series view of weekly engagement (moving average, exponential smoothing — Lecture 8.1)
7. OLAP on a star schema in DuckDB (roll-up, drill-down, slice, dice, pivot — Lectures 2 & 4)

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src")) if (Path.cwd().parent / "src").exists() else sys.path.insert(0, "src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, duckdb
from scipy import stats
from edtech import config as C, features as F, utils as U, viz as V
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)

enr = pd.read_parquet(C.INTERIM / "oulad_enrollments_raw.parquet")
sessions_raw = pd.read_parquet(C.SYNTHETIC / "synthetic_login_sessions.parquet")
subs_raw = pd.read_parquet(C.SYNTHETIC / "synthetic_submission_events.parquet")
# raw (uncleaned) synthetic aggregates - deliberately no deduplication / bot removal here
df = F.integrate(enr, F.session_aggregates(sessions_raw, enr), F.submission_aggregates(subs_raw))
print(df.shape)

(32593, 91)


## 1. Univariate distributions and outliers

In [2]:
num = ["total_clicks", "active_day_fraction", "clicks_per_active_day", "weekly_cv_active_span", "deadline_click_ratio",
       "recency_fraction", "submission_rate", "avg_score_scheduled", "mean_score", "logins_per_week",
       "mean_session_minutes", "video_completion_rate", "forum_posts_per_week", "night_login_share",
       "last_minute_submission_share", "days_to_first_login"]
desc = df[num].describe(percentiles=[.05, .25, .5, .75, .95]).T
desc["skew"] = df[num].skew()
desc["missing_pct"] = 100 * df[num].isna().mean()
U.save_table(desc.round(3).reset_index(names="feature"), "m2_descriptive_stats"); desc.round(2)

,count,mean,std,min,5%,25%,50%,75%,95%,max,skew,missing_pct
total_clicks,32593.0,1149.24,1621.05,0.00,0.0,119.00,557.00,1495.00,4375.00,23481.00,3.04,0.00
active_day_fraction,32593.0,0.20,0.20,0.00,0.0,0.03,0.14,0.32,0.62,1.00,1.17,0.00
clicks_per_active_day,32593.0,17.01,12.90,0.00,0.0,9.00,14.89,23.38,40.00,221.20,1.53,0.00
weekly_cv_active_span,32593.0,1.12,0.72,0.00,0.0,0.75,1.12,1.52,2.30,5.47,0.48,0.00
deadline_click_ratio,32593.0,2.27,2.47,0.00,0.0,0.41,1.61,3.17,7.36,22.50,1.87,0.00
recency_fraction,32593.0,0.38,0.39,0.00,0.0,0.04,0.13,0.81,1.00,1.00,0.56,0.00
submission_rate,32593.0,0.59,0.42,0.00,0.0,0.11,0.78,1.00,1.00,1.00,-0.33,0.00
avg_score_scheduled,32593.0,44.54,34.44,0.00,0.0,6.00,50.58,77.33,90.62,100.00,-0.09,0.00
mean_score,25536.0,72.97,16.45,0.00,42.0,65.00,76.27,84.80,93.00,100.00,-1.29,21.65
logins_per_week,32593.0,2.10,2.66,0.00,0.0,0.33,1.39,3.08,6.34,44.71,5.19,0.00


In [3]:
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
for ax, c in zip(axes.flat, num):
    x = df[c].dropna()
    ax.hist(x.clip(upper=x.quantile(0.995)), bins=50, color=V.CATEGORICAL[0])
    ax.set_title(c, fontsize=10); ax.set_ylabel("enrollments")
fig.suptitle("Distributions (upper 0.5% clipped for display only)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(); V.save(fig, "m2_distributions"); plt.show()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_16484\3936746981.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); V.save(fig, "m2_distributions"); plt.show()


Activity counts are strongly right-skewed (skew of total clicks ≈ 3.0): a log or Yeo-Johnson transform is needed
before distance-based clustering (Lecture 7: *spreading the histogram*). Scores are left-skewed with a spike at 0
for learners who never submitted.

### Outlier detection: boxplot (IQR) and z-score rules

In [4]:
rows = []
for c in num:
    x = df[c].dropna()
    q1, q3 = x.quantile([.25, .75]); iqr = q3 - q1
    z = (x - x.mean()) / x.std()
    rows.append({"feature": c, "mild_outliers_1.5IQR": int(((x < q1 - 1.5*iqr) | (x > q3 + 1.5*iqr)).sum()),
                 "extreme_outliers_3IQR": int(((x < q1 - 3*iqr) | (x > q3 + 3*iqr)).sum()),
                 "abs_z_gt_3": int((z.abs() > 3).sum()), "max": x.max()})
outl = pd.DataFrame(rows); U.save_table(outl, "m2_outliers"); outl

,feature,mild_outliers_1.5IQR,extreme_outliers_3IQR,abs_z_gt_3,max
0,total_clicks,2562,811,671,23481.000000
1,active_day_fraction,693,0,379,1.000000
2,clicks_per_active_day,1019,176,371,221.200000
3,weekly_cv_active_span,777,88,259,5.469513
4,deadline_click_ratio,1675,247,607,22.500000
5,recency_fraction,0,0,0,1.000000
6,submission_rate,0,0,0,1.000000
7,avg_score_scheduled,0,0,0,100.000000
8,mean_score,804,146,377,100.000000
9,logins_per_week,1035,186,254,44.714286


In [5]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
a = df[df.no_activity == 0]
ax[0].scatter(a.active_day_fraction, a.logins_per_week, s=4, alpha=0.25, color=V.CATEGORICAL[0])
ax[0].set(title="Login frequency vs real active days (raw log)", xlabel="active-day fraction (OULAD)", ylabel="logins per week (synthetic, raw)")
sus = a[a.logins_per_week > a.logins_per_week.quantile(0.995)]
ax[0].scatter(sus.active_day_fraction, sus.logins_per_week, s=14, color=V.CATEGORICAL[7], label="top 0.5% login rate")
ax[0].legend()
m = sessions_raw["session_minutes"]
ax[1].hist(m.clip(-50, 300).dropna(), bins=120, color=V.CATEGORICAL[0])
ax[1].set(title="Raw session duration: negatives, zeros and > 24 h values", xlabel="minutes (display clipped to -50..300)", ylabel="sessions", yscale="log")
V.save(fig, "m2_anomalies"); plt.show()
anom = {"negative_minutes": int((m < 0).sum()), "over_1440_minutes": int((m > 1440).sum()), "missing_minutes": int(m.isna().sum()),
        "exact_duplicate_rows": int(sessions_raw.duplicated().sum()),
        "distinct_device_labels": sessions_raw.device_type.dropna().nunique(),
        "enrollments_above_20_logins_per_week": int((df.logins_per_week > 20).sum())}
print(sessions_raw.device_type.value_counts(dropna=False).head(20)); anom

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_16484\211207014.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  V.save(fig, "m2_anomalies"); plt.show()


device_type
desktop       1204002
mobile         871118
tablet         392850
None            27112
Desktop         21021
computer        20901
PC              20889
DESKTOP         20837
 desktop        20720
mobile          15292
Smartphone      15250
Mobile          15118
phone           15104
MOBILE          15063
Tablet           8621
tab              8607
TABLET           8568
iPad             8497
Name: count, dtype: int64


{'negative_minutes': 5412,
 'over_1440_minutes': 2724,
 'missing_minutes': 54046,
 'exact_duplicate_rows': 21505,
 'distinct_device_labels': 17,
 'enrollments_above_20_logins_per_week': 88}

**Anomalies found (input to Module 3):** a small group of enrollments with login rates far above what their real
activity explains (bot-like), negative and > 24 h durations, duplicate rows and 15+ spellings of three device types.

## 2. Correlations among engagement and performance variables
Spearman (rank) correlation is used because distributions are skewed and relationships monotone but non-linear.

In [6]:
corr_cols = ["total_clicks", "active_day_fraction", "logins_per_week", "clicks_per_active_day", "weekly_cv_active_span",
             "deadline_click_ratio", "recency_fraction", "submission_rate", "avg_score_scheduled", "on_time_rate_scheduled",
             "video_completion_rate", "forum_posts_per_week", "night_login_share", "last_minute_submission_share"]
cm = df[df.no_activity == 0][corr_cols].corr("spearman")
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm, cmap=V.DIVERGING, vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)), corr_cols, rotation=60, ha="right"); ax.set_yticks(range(len(corr_cols)), corr_cols)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{cm.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7, color=V.TEXT)
ax.grid(False); fig.colorbar(im, shrink=0.7, label="Spearman ρ")
ax.set_title("Spearman correlations (learners with activity)"); V.save(fig, "m2_correlation_heatmap"); plt.show()
U.save_table(cm.round(3).reset_index(names="feature"), "m2_spearman")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_16484\3741861791.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_title("Spearman correlations (learners with activity)"); V.save(fig, "m2_correlation_heatmap"); plt.show()


,feature,total_clicks,active_day_fraction,logins_per_week,clicks_per_active_day,weekly_cv_active_span,deadline_click_ratio,recency_fraction,submission_rate,avg_score_scheduled,on_time_rate_scheduled,video_completion_rate,forum_posts_per_week,night_login_share,last_minute_submission_share
0,total_clicks,1.000,0.917,0.910,0.615,-0.144,0.007,-0.717,0.690,0.698,0.645,0.385,0.654,-0.038,0.127
1,active_day_fraction,0.917,1.000,0.978,0.293,-0.215,-0.012,-0.799,0.738,0.746,0.656,0.443,0.665,-0.035,0.152
2,logins_per_week,0.910,0.978,1.000,0.312,-0.202,-0.006,-0.782,0.724,0.732,0.645,0.429,0.652,-0.028,0.150
3,clicks_per_active_day,0.615,0.293,0.312,1.000,-0.001,0.023,-0.185,0.214,0.230,0.256,0.075,0.306,-0.033,-0.015
4,weekly_cv_active_span,-0.144,-0.215,-0.202,-0.001,1.000,0.368,-0.098,0.096,0.030,0.042,-0.216,-0.230,0.148,0.203
5,deadline_click_ratio,0.007,-0.012,-0.006,0.023,0.368,1.000,-0.082,0.103,0.056,0.022,-0.153,-0.051,0.200,0.415
6,recency_fraction,-0.717,-0.799,-0.782,-0.185,-0.098,-0.082,1.000,-0.780,-0.760,-0.650,-0.368,-0.507,-0.012,-0.185
7,submission_rate,0.690,0.738,0.724,0.214,0.096,0.103,-0.780,1.000,0.898,0.848,0.376,0.440,0.017,0.229
8,avg_score_scheduled,0.698,0.746,0.732,0.230,0.030,0.056,-0.760,0.898,1.000,0.789,0.427,0.483,-0.001,0.160
9,on_time_rate_scheduled,0.645,0.656,0.645,0.256,0.042,0.022,-0.650,0.848,0.789,1.000,0.351,0.400,-0.019,0.230


* A **volume/frequency block** (clicks, active days, logins, submission, score) is highly inter-correlated
  (ρ 0.7–0.99): one or two representatives are enough for clustering (Lecture 9: minimal redundancy).
* **Pacing** variables — deadline click ratio, burstiness within the active span, last-minute submissions, night
  logins — are nearly independent of volume (|ρ| < 0.25). They describe *how* someone studies, not *how much*,
  which is exactly what distinguishes a "crammer" from a "steady" learner with similar totals.

## 3. Behaviour across course type, cohort and enrollment period

In [7]:
df["registration_period"] = pd.cut(df.date_registration, [-1000, -60, -15, 0, 1000],
                                   labels=["early (>60 d before)", "standard (15-60 d)", "late (<15 d)", "after start"])
def comp(by):
    g = df.groupby(by, observed=True)
    return g.agg(enrollments=("id_student", "size"), median_active_day_fraction=("active_day_fraction", "median"),
                 median_logins_per_week=("logins_per_week", "median"), mean_avg_score=("avg_score_scheduled", "mean"),
                 mean_last_minute_share=("last_minute_submission_share", "mean"),
                 withdrawal_rate=("is_withdrawn", "mean"), at_risk_rate=("at_risk_outcome", "mean")).round(3)
by_domain, by_module, by_cohort, by_period = comp("course_domain"), comp("code_module"), comp("code_presentation"), comp("registration_period")
for name, tb in [("domain", by_domain), ("module", by_module), ("cohort", by_cohort), ("registration_period", by_period)]:
    U.save_table(tb.reset_index(), f"m2_by_{name}")
by_domain

,enrollments,median_active_day_fraction,median_logins_per_week,mean_avg_score,mean_last_minute_share,withdrawal_rate,at_risk_rate
course_domain,,,,,,,
STEM,21402,0.178,1.714,43.208,0.178,0.343,0.552
Social Sciences,11191,0.106,1.000,47.075,0.132,0.251,0.482


In [8]:
by_cohort

,enrollments,median_active_day_fraction,median_logins_per_week,mean_avg_score,mean_last_minute_share,withdrawal_rate,at_risk_rate
code_presentation,,,,,,,
2013B,4684,0.187,1.800,45.285,0.138,0.288,0.553
2013J,8845,0.149,1.436,47.192,0.176,0.268,0.494
2014B,7804,0.124,1.200,40.794,0.153,0.335,0.570
2014J,11260,0.137,1.342,44.731,0.167,0.340,0.515


In [9]:
by_period

,enrollments,median_active_day_fraction,median_logins_per_week,mean_avg_score,mean_last_minute_share,withdrawal_rate,at_risk_rate
registration_period,,,,,,,
early (>60 d before),15482,0.137,1.316,42.166,0.142,0.358,0.551
standard (15-60 d),15549,0.153,1.486,46.882,0.182,0.274,0.507
late (<15 d),1283,0.122,1.205,46.515,0.163,0.210,0.497
after start,234,0.109,1.015,42.948,0.165,0.209,0.496


In [10]:
tests = []
for grp in ["course_domain", "code_module", "code_presentation", "registration_period", "start_month"]:
    for v in ["active_day_fraction", "avg_score_scheduled", "last_minute_submission_share"]:
        samples = [s[v].dropna() for _, s in df.groupby(grp, observed=True)]
        h, p = stats.kruskal(*samples)
        n = sum(len(s) for s in samples); k = len(samples)
        tests.append({"grouping": grp, "variable": v, "kruskal_H": h, "p_value": p, "epsilon_squared": (h - k + 1) / (n - k)})
tests = pd.DataFrame(tests).round(4); U.save_table(tests, "m2_kruskal_tests"); tests

,grouping,variable,kruskal_H,p_value,epsilon_squared
0,course_domain,active_day_fraction,837.1539,0.0000,0.0257
1,course_domain,avg_score_scheduled,39.8055,0.0000,0.0012
2,course_domain,last_minute_submission_share,134.5140,0.0000,0.0041
3,code_module,active_day_fraction,1648.4237,0.0000,0.0504
4,code_module,avg_score_scheduled,843.3395,0.0000,0.0257
5,code_module,last_minute_submission_share,1197.4562,0.0000,0.0366
6,code_presentation,active_day_fraction,227.3292,0.0000,0.0069
7,code_presentation,avg_score_scheduled,135.1311,0.0000,0.0041
8,code_presentation,last_minute_submission_share,39.4562,0.0000,0.0011
9,registration_period,active_day_fraction,91.6893,0.0000,0.0027


In [11]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
mods = sorted(df.code_module.unique())
ax[0].boxplot([df.loc[df.code_module == m, "active_day_fraction"] for m in mods], tick_labels=mods, showfliers=False)
ax[0].set(title="Active-day fraction by module", ylabel="share of course days active")
cohorts = ["2013B", "2013J", "2014B", "2014J"]
w = by_cohort.reindex(cohorts)
ax[1].bar(cohorts, w.withdrawal_rate, color=V.CATEGORICAL[7], width=0.6)
ax[1].set(title="Withdrawal rate by cohort", ylabel="share withdrawn")
p = by_period
ax[2].bar(range(len(p)), p.at_risk_rate, color=V.CATEGORICAL[1], width=0.6)
ax[2].set_xticks(range(len(p)), p.index, rotation=15); ax[2].set(title="At-risk rate by registration timing", ylabel="share Fail + Withdrawn")
fig.tight_layout(); V.save(fig, "m2_group_comparisons"); plt.show()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_16484\2966095295.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); V.save(fig, "m2_group_comparisons"); plt.show()


* **Course type:** STEM learners are active on more days (median 17.8% vs 10.6% of course days) yet are more
  often at risk (55% vs 48%) — STEM modules demand more activity, so raw activity must be read in course context.
  Module is the grouping with the largest (still small, ε² ≈ 0.05) effect on activity.
* **Cohort:** withdrawal rose from 27–29% in the 2013 presentations to 34% in 2014 — a real temporal shift that
  the monitoring module (Module 7) must be able to detect.
* **Enrollment period:** learners who register very early (> 60 days before start) withdraw *more* often (35.8%)
  than standard registrants (27.4%), consistent with speculative early sign-ups; the small late-registration
  group is not more at risk. All group differences are statistically significant (Kruskal–Wallis p < 0.05) but the
  effect sizes are small (ε² ≤ 0.05): individual behaviour varies far more *within* groups than *between* them —
  which is exactly why behavioural segmentation is needed rather than course- or cohort-level policies.

## 4. Required relationships
*Login frequency vs final score* and *video completion vs submission timeliness*.

In [12]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
a = df[(df.no_activity == 0) & (df.logins_per_week < 20)]
hb = ax[0].hexbin(a.logins_per_week, a.avg_score_scheduled, gridsize=45, cmap=V.SEQUENTIAL, mincnt=1, bins="log")
bins = pd.cut(a.logins_per_week, np.arange(0, 12.5, 0.5))
med = a.groupby(bins, observed=True)["avg_score_scheduled"].median()
ax[0].plot([b.mid for b in med.index], med.values, color=V.CATEGORICAL[1], lw=2.5, label="median score")
ax[0].set(title="Login frequency vs score (all scheduled assessments)", xlabel="logins per week", ylabel="average score (non-submission = 0)", xlim=(0, 12))
ax[0].legend(); fig.colorbar(hb, ax=ax[0], label="enrollments (log)")
b = df[(df.submitted_assessments > 0) & df.median_hours_before_deadline.between(-72, 240)]
hb2 = ax[1].hexbin(b.video_completion_rate, b.median_hours_before_deadline, gridsize=45, cmap=V.SEQUENTIAL, mincnt=1, bins="log")
bins = pd.cut(b.video_completion_rate, np.linspace(0, 1, 21))
med2 = b.groupby(bins, observed=True)["median_hours_before_deadline"].median()
ax[1].plot([x.mid for x in med2.index], med2.values, color=V.CATEGORICAL[1], lw=2.5, label="median lead time")
ax[1].axhline(24, color=V.TEXT2, ls=":", lw=1)
ax[1].set(title="Video completion vs submission timeliness", xlabel="video completion rate", ylabel="median hours before deadline")
ax[1].legend(); fig.colorbar(hb2, ax=ax[1], label="enrollments (log)")
V.save(fig, "m2_required_relationships"); plt.show()
rel = {"rho_logins_score": a[["logins_per_week", "avg_score_scheduled"]].corr("spearman").iloc[0, 1],
       "rho_video_leadtime": b[["video_completion_rate", "median_hours_before_deadline"]].corr("spearman").iloc[0, 1],
       "rho_logins_exam": df[["logins_per_week", "exam_score"]].dropna().corr("spearman").iloc[0, 1]}
rel

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_16484\290893715.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  V.save(fig, "m2_required_relationships"); plt.show()


{'rho_logins_score': np.float64(0.7358517998464644),
 'rho_video_leadtime': np.float64(0.13724610140887616),
 'rho_logins_exam': np.float64(0.21833736031657724)}

Score rises steeply up to about 3–4 logins per week (median ≈ 75–80) and then flattens — diminishing returns
(overall ρ ≈ 0.74). Getting low-frequency learners to a *regular minimum* matters more than pushing frequent
learners further. Video completion is only weakly related to submission lead time (ρ ≈ 0.14): the typical
learner submits within the last 0–24 hours *whatever* their video completion, so timeliness is a separate
behavioural dimension, not a by-product of engagement. (Values above 1 on the x-axis are the injected
"completed > started" defect, removed in Module 3.)

## 5. Behaviour vs final outcome

In [13]:
order = ["Distinction", "Pass", "Fail", "Withdrawn"]
out = df.groupby("final_result")[["active_day_fraction", "logins_per_week", "avg_score_scheduled", "video_completion_rate",
                                  "forum_posts_per_week", "last_minute_submission_share", "deadline_click_ratio",
                                  "recency_fraction", "no_activity"]].median().reindex(order)
out.insert(0, "enrollments", df.final_result.value_counts().reindex(order))
U.save_table(out.round(3).reset_index(), "m2_by_outcome"); out.round(3)

,enrollments,active_day_fraction,logins_per_week,avg_score_scheduled,video_completion_rate,forum_posts_per_week,last_minute_submission_share,deadline_click_ratio,recency_fraction,no_activity
final_result,,,,,,,,,,
Distinction,3024,0.381,3.629,88.455,0.771,0.154,0.000,1.508,0.030,0.0
Pass,12361,0.289,2.769,75.400,0.700,0.077,0.125,1.908,0.047,0.0
Fail,7052,0.078,0.769,21.697,0.563,0.000,0.000,2.182,0.458,0.0
Withdrawn,10156,0.019,0.179,0.000,0.414,0.000,0.000,0.000,0.907,0.0


## 6. Weekly engagement as a time series (Lecture 8.1)
Average clicks per enrollment per course week, by final outcome, with a 3-week **moving average** and
**exponential smoothing** (α = 0.3). Vertical lines mark TMA deadlines of module FFF 2014J as an example.

In [14]:
day = pd.read_parquet(C.INTERIM / "enrollment_day.parquet", columns=C.KEY + ["date", "clicks"])
day = day[day.date.between(0, 270)].merge(df[C.KEY + ["final_result"]], on=C.KEY)
day["week"] = day.date // 7
n_by = df.final_result.value_counts()
ws = day.groupby(["final_result", "week"]).clicks.sum().unstack(0).div(n_by, axis=1).fillna(0)
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
for o in order:
    s = ws[o]
    ax[0].plot(s.index, s.ewm(alpha=0.3).mean(), color=V.OUTCOME_COLORS[o], label=o)
ax[0].set(title="Weekly clicks per enrollment (exponential smoothing, α=0.3)", xlabel="course week", ylabel="clicks / enrollment")
ax[0].legend()
f = day[(day.code_module == "FFF") & (day.code_presentation == "2014J")].groupby("date").clicks.sum()
f = f.reindex(range(0, 270), fill_value=0) / (df.code_module.eq("FFF") & df.code_presentation.eq("2014J")).sum()
ax[1].plot(f.index, f.values, color=V.GRID, lw=1, label="daily")
ax[1].plot(f.index, f.rolling(7, center=True).mean(), color=V.CATEGORICAL[0], label="7-day moving average")
asm = pd.read_csv(C.RAW / "assessments.csv", na_values="?")
for d in asm[(asm.code_module == "FFF") & (asm.code_presentation == "2014J") & (asm.assessment_type == "TMA")].date.dropna():
    ax[1].axvline(d, color=V.CATEGORICAL[1], ls="--", lw=1)
ax[1].set(title="FFF 2014J daily clicks per enrollment; dashed = TMA deadlines", xlabel="course day", ylabel="clicks / enrollment")
ax[1].legend(); V.save(fig, "m2_time_series"); plt.show()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_16484\3522661115.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax[1].legend(); V.save(fig, "m2_time_series"); plt.show()


The series has a **downward trend** (attrition), a **"seasonal" component tied to the assessment calendar**
(peaks right before each TMA deadline) and irregular noise. The deadline peaks are the population-level trace of
the cramming behaviour measured per learner by `deadline_click_ratio`. Learners who later withdraw or fail are
already less than half as active as passers in **week 0–2**, so the gap is visible long before the outcome —
the window in which intervention must happen.

## 7. OLAP on a star schema (DuckDB, column store)
A small dimensional model makes multi-dimensional questions one SQL query away:

* **fact_enrollment** — measures: clicks, active days, logins, posts, score, at-risk flag
* **dim_learner** (age band, education, IMD, gender, disability) · **dim_course** (module, domain) ·
  **dim_presentation** (year, start month) · **dim_registration** (registration period)

DuckDB stores tables column-wise, which suits these scan-and-aggregate queries (Lecture 4).

In [15]:
con = duckdb.connect(str(C.WAREHOUSE / "learning_dw.duckdb"))
dim_learner = df[["id_student", "gender", "age_band", "highest_education", "imd_band", "disability", "region"]].drop_duplicates("id_student")
dim_course = df[["code_module", "course_domain"]].drop_duplicates()
dim_presentation = df[["code_presentation", "presentation_year", "start_month"]].drop_duplicates()
fact = df[C.KEY + ["registration_period", "final_result", "total_clicks", "active_days", "logins_total", "forum_posts",
                   "avg_score_scheduled", "at_risk_outcome", "is_withdrawn"]].astype({"registration_period": str})
for name, tb in [("dim_learner", dim_learner), ("dim_course", dim_course), ("dim_presentation", dim_presentation), ("fact_enrollment", fact)]:
    con.register("tmp", tb); con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM tmp"); con.unregister("tmp")
con.execute("SELECT table_name, estimated_size FROM duckdb_tables()").df()

,table_name,estimated_size
0,dim_course,7
1,dim_learner,28785
2,dim_presentation,4
3,fact_enrollment,32593


In [16]:
# ROLL-UP: module -> domain -> all (GROUP BY ROLLUP)
rollup = con.execute("""
SELECT c.course_domain, f.code_module, COUNT(*) AS enrollments,
       ROUND(AVG(f.at_risk_outcome), 3) AS at_risk_rate, ROUND(AVG(f.total_clicks)) AS avg_clicks
FROM fact_enrollment f JOIN dim_course c USING (code_module)
GROUP BY ROLLUP (c.course_domain, f.code_module) ORDER BY 1 NULLS LAST, 2 NULLS LAST""").df()
rollup

,course_domain,code_module,enrollments,at_risk_rate,avg_clicks
0,STEM,CCC,4434,0.622,1006.0
1,STEM,DDD,6272,0.584,804.0
2,STEM,EEE,2934,0.438,1289.0
3,STEM,FFF,7762,0.530,2163.0
4,STEM,None,21402,0.552,1405.0
5,Social Sciences,AAA,748,0.290,1515.0
6,Social Sciences,BBB,7909,0.525,629.0
7,Social Sciences,GGG,2534,0.403,502.0
8,Social Sciences,None,11191,0.482,660.0
9,None,None,32593,0.528,1149.0


In [17]:
# DRILL-DOWN + DICE: STEM courses, October presentations, by age band and education
dice = con.execute("""
SELECT l.age_band, l.highest_education, COUNT(*) AS n, ROUND(AVG(f.at_risk_outcome), 3) AS at_risk_rate,
       ROUND(AVG(f.logins_total), 1) AS avg_logins
FROM fact_enrollment f JOIN dim_learner l USING (id_student) JOIN dim_course c USING (code_module)
JOIN dim_presentation p USING (code_presentation)
WHERE c.course_domain = 'STEM' AND p.start_month = 'October'
GROUP BY 1, 2 HAVING COUNT(*) >= 50 ORDER BY at_risk_rate DESC""").df()
dice.head(10)

,age_band,highest_education,n,at_risk_rate,avg_logins
0,0-35,No Formal quals,88,0.693,50.6
1,0-35,Lower Than A Level,3603,0.644,76.4
2,35-55,Lower Than A Level,1165,0.575,103.9
3,0-35,A Level or Equivalent,4723,0.496,84.6
4,35-55,A Level or Equivalent,1148,0.473,109.3
5,0-35,HE Qualification,1262,0.448,92.8
6,35-55,HE Qualification,862,0.432,114.4
7,35-55,Post Graduate Qualification,93,0.409,97.6
8,0-35,Post Graduate Qualification,63,0.333,106.5
9,55<=,HE Qualification,65,0.323,150.0


In [18]:
# SLICE (one presentation) + PIVOT (registration period x outcome)
pivot = con.execute("""
PIVOT (SELECT registration_period, final_result FROM fact_enrollment WHERE code_presentation = '2014J')
ON final_result USING COUNT(*) GROUP BY registration_period ORDER BY registration_period""").df()
con.close(); U.save_table(rollup, "m2_olap_rollup"); U.save_table(pivot, "m2_olap_pivot"); pivot

,registration_period,Distinction,Fail,Pass,Withdrawn
0,after start,11,30,62,28
1,early (>60 d before),513,825,2001,2022
2,late (<15 d),78,173,286,171
3,nan,0,0,1,8
4,standard (15-60 d),562,949,1943,1597


## Key EDA insights
1. **Two independent behavioural axes** exist: *volume/regularity* (clicks, active days, logins, submissions,
   score) and *pacing* (deadline concentration, burstiness, last-minute submission, night study). Segmenting on
   both is what allows personas such as "last-minute crammer" vs "steady learner".
2. **Engagement relates to performance with diminishing returns** — the biggest score gains are between 0 and
   ~3 logins per week.
3. **Risk is visible early**: eventually-withdrawn learners are less than half as active as passers from week 0;
   withdrawal rose from 2013 to 2014 cohorts (a drift to monitor).
4. **Course context matters** (module ε² for activity), so profiles are always reported with their module mix.
5. **Data issues** to fix: duplicates, device label inconsistency, impossible durations, bot-like login counts,
   and 4,093 enrollments without any in-window VLE activity (treated as a rule-based segment, not as noise).

In [19]:
U.save_metrics("m2", {"anomalies": anom, "relationships": rel, "skew_total_clicks": float(desc.loc["total_clicks", "skew"]),
                      "n_no_activity": int(df.no_activity.sum()),
                      "by_cohort_withdrawal": by_cohort.withdrawal_rate.to_dict(),
                      "by_period_at_risk": {str(k): v for k, v in by_period.at_risk_rate.items()},
                      "by_domain": by_domain.reset_index().to_dict("records")})